# Setup 1 — teste de fluxos (smoke test)

**Objetivo:** provar que **cada fluxo do pipeline funciona 100%** antes de puxar histórico.
Cada bloco roda **um fluxo no menor período possível** (1 dia, `D` = último dia útil **antes de
hoje**, para a fonte já estar publicada) e, em seguida, **confere no `.db`** que aquele fluxo
realmente gravou linhas (`[OK]` / `[VAZIO]`).

**Ordem:** rode este notebook **primeiro** → depois `setup_2_carga.ipynb` (histórico) →
depois o dia a dia em `pipeline.ipynb`.

**Pode dar `Run All`.** Nada aborta: cada bloco mostra `[OK]`/`[FALHA]` do fluxo e `[OK]`/`[VAZIO]`
da conferência. A célula final resume o que gravou. **Rode a partir da pasta `code/`.**

## Bloco 0 — cria o `.db` + config (rodar primeiro)

In [ ]:
import sys
from pathlib import Path
from datetime import date
import sqlite3

if not (Path.cwd() / "scripts").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))            # p/ 'import lib.*'
sys.path.insert(0, str(Path.cwd() / "scripts"))  # p/ 'import pipeline_core'
import pipeline_core as pc
from lib.db import get_db

# 1) cria o .db e o schema (idempotente — nao apaga dados existentes)
get_db().close()
DB = "data/trades.db"
print("Base criada/verificada:", Path(DB).resolve())

# 2) dia de teste: ultimo dia util ANTES de hoje (garante fonte ja publicada;
#    rodando de manha, o arquivo de HOJE ainda nao saiu) e o anterior (Dant)
D    = pc.dia_util_anterior(date.today()).isoformat()
Dant = pc.dia_util_anterior(D).isoformat()
print("Dia de teste D =", D, "| Dant =", Dant)

# 3) helper: confere que o fluxo gravou linhas no .db
_smoke = []
def checar(desc, sql, params=()):
    try:
        n = sqlite3.connect(DB).execute(sql, params).fetchone()[0]
    except Exception as e:
        print(f"[ERRO ] {desc}: {e}"); _smoke.append((desc, False)); return False
    ok = bool(n and n > 0)
    print(f"{'[OK]    ' if ok else '[VAZIO] '}{desc}: {n:,} linha(s)")
    _smoke.append((desc, ok)); return ok

## Fluxos — 1 bloco por fluxo (rodam em ordem no `Run All`)
Cada bloco: roda o fluxo p/ o menor período → confere no `.db`. Os fluxos de **cálculo**
dependem dos de **scraping** acima (por isso a ordem).

In [ ]:
# 1. Boletim B3 (negocios) -> NegociosBrutos
pc.boletim(Dant, D)
checar("boletim -> NegociosBrutos",
       "SELECT COUNT(*) FROM NegociosBrutos WHERE dtNegocio IN (?,?)", (Dant, D))

In [ ]:
# 2. Anbima debentures (indicativas) -> AnbimaIndicativos (+ InfoAtivos)
pc.anbima_deb(Dant, D)
checar("anbima_deb -> AnbimaIndicativos",
       "SELECT COUNT(*) FROM AnbimaIndicativos WHERE dtReferencia IN (?,?)", (Dant, D))

In [ ]:
# 3. Anbima CRI/CRA (indicativas, Playwright) -> AnbimaIndicativos
pc.anbima_cricra(D)
checar("anbima_cricra -> AnbimaIndicativos (dia D)",
       "SELECT COUNT(*) FROM AnbimaIndicativos WHERE dtReferencia = ?", (D,))

In [ ]:
# 4. FI Analytics planilha (caracteristicas, Playwright+login) -> InfoAtivos
pc.fianalytics()
checar("fianalytics -> InfoAtivos", "SELECT COUNT(*) FROM InfoAtivos")

In [ ]:
# 5. Anbima Data (caracteristicas + fluxo, Playwright) -> InfoAtivos + FluxoAtivos
#    modo incremental (menor periodo) — depende do boletim (bloco 1)
pc.anbima_data(Dant, D)
checar("anbima_data -> FluxoAtivos", "SELECT COUNT(*) FROM FluxoAtivos")

In [ ]:
# 6. Anbima NTN-B (MtM) -> MtmAnbima (tickers NTN-B). Duration paralela + skip.
pc.ntnb(Dant, D)
checar("ntnb -> MtmAnbima (NTN-B)",
       "SELECT COUNT(*) FROM MtmAnbima WHERE cdTicker LIKE 'NTN-B%' AND dtReferencia IN (?,?)", (Dant, D))

In [ ]:
# 7. Curva DI B3 (MtM) -> MtmAnbima (tickers DI1F..)
pc.curva_di(Dant); pc.curva_di(D)
checar("curva_di -> MtmAnbima (DI)",
       "SELECT COUNT(*) FROM MtmAnbima WHERE cdTicker LIKE 'DI1%' AND dtReferencia IN (?,?)", (Dant, D))

In [ ]:
# 8. Outstanding via Bloomberg -> Outstanding. SO NO BANCO.
#    No PC pessoal da [FALHA]/[VAZIO] (sem terminal Bloomberg) — normal.
pc.outstanding(D)
checar("outstanding -> Outstanding (so no banco)",
       "SELECT COUNT(*) FROM Outstanding WHERE dtOutstanding = ?", (D,))

In [ ]:
# 9. Calcular taxa por trade (cascata FI Analytics -> B3) -> NegociosProcessados
pc.calc_taxa(D)
checar("calc_taxa -> NegociosProcessados (liquidacao D)",
       "SELECT COUNT(*) FROM NegociosProcessados WHERE dtLiquidacao = ?", (D,))

In [ ]:
# 10. Filtrar (VALIDO/FUNDO/BROKER/PF) -> NegociosProcessados.cdStatus
pc.filtrar(D)
checar("filtrar -> cdStatus preenchido (liquidacao D)",
       "SELECT COUNT(*) FROM NegociosProcessados WHERE dtLiquidacao = ? AND cdStatus IS NOT NULL", (D,))

In [ ]:
# 11. Spread Anbima das indicativas -> AnbimaIndicativos.vrSpreadAnbima
pc.spread_anbima(Dant); pc.spread_anbima(D)
checar("spread_anbima -> vrSpreadAnbima preenchido",
       "SELECT COUNT(*) FROM AnbimaIndicativos WHERE dtReferencia IN (?,?) AND vrSpreadAnbima IS NOT NULL", (Dant, D))

In [ ]:
# 12. Match de referencia (global, sem data) -> InfoAtivos.cdReferencia
pc.match_ref()
checar("match_ref -> cdReferencia preenchido",
       "SELECT COUNT(*) FROM InfoAtivos WHERE cdReferencia IS NOT NULL")

In [ ]:
# 13. Spread over dos trades (casado por dtNegocio) -> NegociosProcessados.vrSpreadOver
pc.spread_over(D)
checar("spread_over -> vrSpreadOver preenchido (liquidacao D)",
       "SELECT COUNT(*) FROM NegociosProcessados WHERE dtLiquidacao = ? AND vrSpreadOver IS NOT NULL", (D,))

In [ ]:
# 14. Gerar relatorio HTML (toda a base)
pc.relatorio()
htmls = sorted(Path("data/relatorios").glob("*.html"))
ok = len(htmls) > 0
_smoke.append(("relatorio -> HTML gerado", ok))
print(f"{'[OK]    ' if ok else '[VAZIO] '}relatorio: {len(htmls)} arquivo(s) em data/relatorios")
if htmls: print("  ultimo:", htmls[-1].name)

## Resumo do smoke test

In [ ]:
oks = sum(1 for _, ok in _smoke if ok)
print(f"{'#'*60}\n# SMOKE TEST: {oks}/{len(_smoke)} fluxos gravaram no .db\n{'#'*60}")
for desc, ok in _smoke:
    print(f"  {'OK   ' if ok else 'VAZIO'}  {desc}")
faltou = [d for d, ok in _smoke if not ok]
if faltou:
    print("\nVAZIO = fluxo rodou mas nao gravou p/ o dia testado. Verifique:")
    for d in faltou: print("  -", d)
    print("\nObs: 'outstanding' fica VAZIO fora do banco (sem Bloomberg) — normal.")
else:
    print("\nTudo certo — todos os fluxos gravaram no .db.")